# Data Preprocessing: Mapping Raw Variables into Analytical Groups

<div style="font-size: 10px;">

| Raw Variable | New Variable  | 
|--------------|---------------|
| SEX          | SEX_GROUP     |
| AGE          | AGE_GROUP     |
| RACE         | RACE_GROUP    |
| EDUCD        | EDU_LEVEL     |
| WORKEDYR     | WORK_RECENCY  |

</div>


### 0. Reading IPUMS Data

- The IPUMS USA microdata extract is provided in a fixed-width format (FWF), where each variable occupies predefined character positions in each row.

- The following code loads the raw .dat file into a pandas DataFrame by explicitly specifying column boundaries (colspecs) and assigning readable variable names.

**[Table] Fixed-Width Column Specifications**

<div style="font-size: 10px;">

| Variable    | Type | Columns  | Width | Included (2023) |
|-------------|------|----------|--------|------------------|
| YEAR        | H    | 1–4      | 4      | Yes              |
| SAMPLE      | H    | 5–10     | 6      | Yes              |
| SERIAL      | H    | 11–18    | 8      | Yes              |
| STATEFIP    | H    | 19–20    | 2      | Yes              |
| PERNUM      | P    | 21–24    | 4      | Yes              |
| SEX         | P    | 25       | 1      | Yes              |
| AGE         | P    | 26–28    | 3      | Yes              |
| RACE        | P    | 29       | 1      | Yes              |
| RACED       | P    | 30–32    | 3      | Yes              |
| EDUC        | P    | 33–34    | 2      | Yes              |
| EDUCD       | P    | 35–37    | 3      | Yes              |
| DEGFIELD    | P    | 38–39    | 2      | Yes              |
| DEGFIELDD   | P    | 40–43    | 4      | Yes              |
| DEGFIELD2   | P    | 44–45    | 2      | Yes              |
| DEGFIELD2D  | P    | 46–49    | 4      | Yes              |
| EMPSTAT     | P    | 50       | 1      | Yes              |
| EMPSTATD    | P    | 51–52    | 2      | Yes              |
| LABFORCE    | P    | 53       | 1      | Yes              |
| OCC         | P    | 54–57    | 4      | Yes              |
| WORKEDYR    | P    | 58       | 1      | Yes              |

</div>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import json
import random

colspecs = [
    (0, 4),     # YEAR (1-4)
    (4, 10),     # SAMPLE (5-10)
    (10, 18),    # SERIAL (11-18)
    (18, 20),    # STATEFIP (19-20)
    (20, 24),    # PERNUM (21-24)
    (24, 25),     # SEX (25)
    (25, 28),    # AGE (26-28)
    (28, 29),    # RACE (29)
    (29, 32),    # RACED (30-32)
    (32, 34),    # EDUC (33-34)
    (34, 37),    # EDUCD (35-37)
    (37, 39),    # DEGFIELD (38-39)
    (39, 43),    # DEGFIELDD (40-43)
    (43, 45),    # DEGFIELD2 (44-45)
    (45, 49),    # DEGFIELD2D (46-49)
    (49, 50),    # EMPSTAT (50)
    (50, 52),    # EMPSTATD (51-52)
    (52, 53),    # LABFORCE (53)
    (53, 57),    # OCC (54-57)
    (57, 58),    # WORKEDYR (58)
]
names = [
    "YEAR", "SAMPLE", "SERIAL", "STATEFIP", "PERNUM", "SEX", "AGE", "RACE",
    "RACED", "EDUC", "EDUCD", "DEGFIELD", "DEGFIELDD", "DEGFIELD2",
    "DEGFIELD2D", "EMPSTAT", "EMPSTATD", "LABFORCE", "OCC", "WORKEDYR"
]

df = pd.read_fwf(
    "./sources/usa_00005.dat", # 0003.dat is another file option which does not have 
    colspecs=colspecs,
    names=names
)

### 0-1. SAMPLE_SERIAL_PERNUM

In [ ]:
df.head()

In [ ]:
# make unique id from SAMPLE, SERIAL, PERNUM
df['SAMPLE_SERIAL_PERNUM'] = df['SAMPLE'].astype(str).str.zfill(6) + "_" + \
                             df['SERIAL'].astype(str).str.zfill(8) + "_" + \
                             df['PERNUM'].astype(str).str.zfill(4)
print(f"Number of rows in df: {len(df)}")
print(f"Number of unique SAMPLE_SERIAL_PERNUM entries: {len(df['SAMPLE_SERIAL_PERNUM'].unique())}")

### 0-2. State or REGION



In [ ]:
def map_statefip_to_name(statefip):
    mapping = {
        1: "Alabama",
        2: "Alaska",
        4: "Arizona",
        5: "Arkansas",
        6: "California",
        8: "Colorado",
        9: "Connecticut",
        10: "Delaware", 
        11: "District of Columbia",
        12: "Florida",
        13: "Georgia",
        15: "Hawaii",
        16: "Idaho",
        17: "Illinois",
        18: "Indiana",
        19: "Iowa",
        20: "Kansas",
        21: "Kentucky",
        22: "Louisiana",
        23: "Maine",
        24: "Maryland",
        25: "Massachusetts",
        26: "Michigan",
        27: "Minnesota",
        28: "Mississippi",
        29: "Missouri",
        30: "Montana",
        31: "Nebraska",
        32: "Nevada",
        33: "New Hampshire",
        34: "New Jersey",
        35: "New Mexico",
        36: "New York",
        37: "North Carolina",
        38: "North Dakota",
        39: "Ohio",
        40: "Oklahoma",
        41: "Oregon",
        42: "Pennsylvania",
        44: "Rhode Island",
        45: "South Carolina",
        46: "South Dakota",
        47: "Tennessee",
        48: "Texas",
        49: "Utah",
        50: "Vermont",
        51: "Virginia",
        53: "Washington",
        54: "West Virginia",
        55: "Wisconsin",
        56: "Wyoming",
        61: "Maine-New Hampshire-Vermont",
        62: "Massachusetts-Rhode Island",
        63: "Minnesota-Iowa-Missouri-Kansas-Nebraska-S.Dakota-N.Dakota",
        64: "Maryland-Delaware",
        65: "Montana-Idaho-Wyoming",
        66: "Utah-Nevada",
        67: "Arizona-New Mexico",
        68: "Alaska-Hawaii",
        72: "Puerto Rico",
        97: "Military/Mil. Reservation",
        99: "State not identified"
    }
    return mapping.get(statefip, "Unknown")

df['STATE_NAME'] = df['STATEFIP'].apply(map_statefip_to_name)
print(f"Number of 61~ STATEFIP entries: {len(df[(df['STATEFIP'] >= 61)])}")
df = df[(df['STATEFIP'] < 61)]
len(df)
print(df["STATE_NAME"].unique())

### 1. Sex (SEX → SEX_GROUP)

- Original SEX codes: 1 = Male, 2 = Female, otherwise treated as missing.
- The function converts raw codes into "Male", "Female", and "Unknown" categories.
- All "Unknown" entries are removed from the dataset.

In [ ]:
def map_sex(sex):
    if sex == 1:
        return "Male"
    elif sex == 2:
        return "Female"
    else:
        return "Unknown"

df["SEX_GROUP"] = df["SEX"].apply(map_sex)
# print the number of unknown sex
print("Number of unknown sex:", (df["SEX_GROUP"] == "Unknown").sum())
# drop unknown sex
df = df[df["SEX_GROUP"] != "Unknown"]

### 2. Age (AGE → AGE_GROUP)

- Age values are grouped into seven standard demographic age ranges.
- Invalid or missing age values are labeled as "Unknown" and removed.

In [ ]:
def map_age(age):
    if age < 18:
        return "Under 18"
    elif 18 <= age <= 24:
        return "18-24"
    elif 25 <= age <= 34:
        return "25-34"
    elif 35 <= age <= 44:
        return "35-44"
    elif 45 <= age <= 54:
        return "45-54"
    elif 55 <= age <= 64:
        return "55-64"
    elif age >= 65:
        return "65+"
    else:
        return "Unknown"

df["AGE_GROUP"] = df["AGE"].apply(map_age)
# print the number of unknown age
print("Number of unknown age:", (df["AGE_GROUP"] == "Unknown").sum())
# drop unknown age
df = df[df["AGE_GROUP"] != "Unknown"]
# print the oldest and youngest age in the dataset
print("Oldest age in dataset:", df["AGE"].max())
print("Youngest age in dataset:", df["AGE"].min())

### 3. Race (RACE → RACE_GROUP)

- Raw race codes are mapped into broader racial categories based on U.S. Census conventions.
- Codes 4–6 are combined into an Asian/Pacific Islander group.
- Rare, ambiguous, or unclassified codes map to "Other/Unknown".

In [ ]:
def map_race(race):
    if race == 1:
        return "White"
    elif race == 2:
        return "Black"
    elif race == 3:
        return "AIAN"
    elif race in [4, 5, 6]:
        return "Asian/Pacific Islander"
    elif race == 8:
        return "Two races"
    elif race == 9:
        return "Three+ races"
    else:
        return "Other/Unknown"

df["RACE_GROUP"] = df["RACE"].apply(map_race)

print("Number of other/unknown race:", (df["RACE_GROUP"] == "Other/Unknown").sum())

df = df[df["RACE_GROUP"] != "Other/Unknown"]

### 4. Education (EDUCD → EDU_LEVEL)

- The EDUCD variable contains numerous detailed codes; this function groups them into seven interpretable education levels:
    - No schooling
    - High school or less
    - Some college / Associate
	- Bachelor’s
	- Master’s
	- Professional degree
	- Doctorate
- Values outside these ranges are labeled "Unknown" and removed.
- The mapping follows the structure of the U.S. Census education classification.

In [ ]:
def map_education(educd):
    if educd in [0, 1, 2]:
        return "No schooling"
    elif educd in range(10, 18): # 10, 11, 12, 13, 14, 15, 16, 17
        return "Primary school or less"
    elif educd in range(20, 27): # 20, 21, 22, 23, 24, 25, 26
        return "Middle school"
    elif educd in range(30, 62): # 30, 40, 50, 60, 61
        return "Some high school"
    elif educd in [62, 63, 64]: # 62, 63, 64
        return "High school graduate/GED"
    elif educd in [65, 70, 71, 80, 90]:
        return "Some college (no degree)"
    elif educd in [81, 82, 83]:
        return "Associate's degree"
    elif educd in [100,101]:
        return "Bachelor's"
    elif educd in [110,111,112,113, 114]:
        return "Master's"
    elif educd == 115:
        return "Professional degree"
    elif educd == 116:
        return "Doctorate"
    else:
        return "Unknown"
    
df["EDU_LEVEL"] = df["EDUCD"].apply(map_education)
# print the number of unknown education
print("Number of unknown education:", (df["EDU_LEVEL"] == "Unknown").sum())
df = df[df["EDU_LEVEL"] != "Unknown"]

### 5. Work Recency (WORKEDYR → WORK_RECENCY)

- This variable encodes how recently an individual has worked:
	- 3 → Worked within the last 12 months
	- 2 → Worked 1–5 years ago
	- 1 → No recent work (5+ years)

In [ ]:
def map_recent_work(x):
    if x == 3:
        return "Worked in past 12 months"
    elif x == 2:
        return "Worked 1-5 years ago"
    elif x == 1:
        return "No recent work (5+ years)"
    else:
        return "Unknown"

df["WORK_RECENCY"] = df["WORKEDYR"].apply(map_recent_work)
# print the number of unknown work recency
print("Number of unknown work recency:", (df["WORK_RECENCY"] == "Unknown").sum())
# drop unknown work recency
df = df[df["WORK_RECENCY"] != "Unknown"]

### 6. Mapping Census Occupation Codes (OCC) to SOC Codes Using Crosswalk

To standardize occupation information, the raw 2018 Census Occupation Codes (OCC) are mapped to the corresponding Standard Occupational Classification (SOC) codes using an official Census crosswalk table.


In [ ]:
crosswalk = pd.read_excel("./sources/census2018_occ_to_soc.xlsx")

df["OCC"] = df["OCC"].astype(str).str.zfill(4) # Census occupation codes are fixed-width numeric identifiers; values like "25" must be stored as "0025" to correctly match crosswalk entries.

crosswalk["2018 Census Code"] = crosswalk["2018 Census Code"].astype(str).str.zfill(4)

df = df.merge(
    crosswalk[["2018 Census Code", "2018 SOC Code"]],
    left_on="OCC",
    right_on="2018 Census Code",
    how="left"
)

df["SOC"] = df["2018 SOC Code"]

# drop "2018 Census Code" and "2018 SOC Code" columns
df = df.drop(columns=["2018 Census Code", "2018 SOC Code"])


print("SOC Missing Rate:",  df["SOC"].isna().mean())
print("SOC Non-missing Count:",  df["SOC"].notna().sum())

df.head()

Identifying the Top 20 SOC Codes and Mapping to Occupation Titles

This section extracts the 20 most frequent SOC codes in the dataset and retrieves their corresponding descriptive occupation titles from the Census SOC crosswalk file.

In [ ]:
crosswalk["SOC"] = crosswalk["2018 SOC Code"].astype(str)

top20_soc = df["SOC"].value_counts().head(20).index.tolist()

top20_info = crosswalk[crosswalk["2018 SOC Code"].isin(top20_soc)][
    ["2018 SOC Code", "U.S. Census Bureau",]
]

top20_info["count"] = top20_info["2018 SOC Code"].map(df["SOC"].value_counts())
top20_info = top20_info.sort_values(by="count", ascending=False)

top20_info

In [ ]:
plt.figure(figsize=(12,4))
df["SOC"].value_counts().head(20).plot(kind="bar")
plt.title("Top 20 SOC Codes by Frequency")
plt.xlabel("SOC")
plt.ylabel("Count")
plt.show()

### 7. Mapping Selected SOC Codes to Job Categories and Filtering the Dataset

- Rows where the SOC code is one of the selected occupations
- Individuals 18–65 years old
- Individuals not recently employed (not worked in past 12 months)

In [ ]:
soc_job_map = {
    "15-1252": ["Software Developers", "software_developers"],
    "47-2061": ["Construction Laborers", "construction_laborers"],
    "25-2020": ["Elementary and Middle School Teachers", "elementary_middle_school_teachers"],
    "29-1141": ["Registered Nurses", "registered_nurses"],
    "13-2011": ["Accountants and Auditors", "accountants_auditors"]
}

# add job title column
df["job"] = df["SOC"].map(lambda x: soc_job_map.get(x, [None, None])[1])
# drop if job is None
df = df[df["job"].notna()]

# drop rows if "WORK_RECENCY" is "Worked in past 12 months"
# df = df[df["WORK_RECENCY"] != "Worked in past 12 months"]

# drop rows with "AGE" bigger than 65 and smaller than 18
# df = df[(df["AGE"] <= 65) & (df["AGE"] >= 18)]
print("Oldest age in dataset:", df["AGE"].max(), "num of oldest:", (df["AGE"] == df["AGE"].max()).sum())
print("Youngest age in dataset:", df["AGE"].min(), "num of youngest:", (df["AGE"] == df["AGE"].min()).sum())

### 9-2. Deterministic SKILLS, TECHSKILLS

In [ ]:
def det_skills(
    occupation_key: str,
    job_spec: dict,
    n_skills: int = 6,
):
    spec = job_spec[occupation_key]
    skills_pool = spec["skills"]
    job_zone = spec["job_zone"]

    sampled_skills = skills_pool[:n_skills]


    return {
        "skills": sampled_skills,
        "techskills": [],
        "certifications": [],
        "job_zone": job_zone
    }

#### load job_spec_deterministic

In [ ]:
def load_job_spec(path="job_spec.json"):
    with open(path, "r", encoding="utf-8") as f:
        job_spec = json.load(f)
    return job_spec

job_spec_det = load_job_spec(path="./sources/job_spec_deterministic.json")

#### Final!!

In [ ]:
def apply_det(row):
    occ_key = soc_job_map[row["SOC"]][1]

    sampled = det_skills(
        occupation_key=occ_key,
        job_spec=job_spec_det,
        n_skills=6,
    )
    return pd.Series(sampled)

df[["skills", "techskills", "certifications", "job_zone"]] = df.apply(apply_det, axis=1)

In [ ]:
# total num for each job
job_counts = df['job'].value_counts()
job_counts

In [ ]:
df['n_skills'] = df['skills'].apply(len)
df['n_techskills'] = df['techskills'].apply(len)
df['n_certifications'] = df['certifications'].apply(len)
# edu_skill_stats = df.groupby(["job", "EDU_LEVEL_INT"]).agg(
#     count=("n_skills", "size"),
#     mean_skills=("n_skills", "mean"),
#     mean_techskills=("n_techskills", "mean"),
#     mean_certifications=("n_certifications", "mean")
# )
# edu_skill_stats

## Save & Analysis

In [ ]:
# save df to csv
df.to_csv("processed_job_data_0102.csv", index=False)